In [178]:
! pip install pymupdf python-docx openpyxl


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


# DOCUMENTS LOAD

In [ ]:
from pathlib import Path

def load_documents_from_dir(directory: str):

    paths = []

    for path in Path(directory).rglob("*"):

        if path.suffix.lower() in {
            ".pdf",
            ".docx",
            ".txt"
        }:
            paths.append(path)

    return paths

In [179]:
import fitz  # pymupdf

def load_pdf_as_text(path: str) -> str:
    """
    Загружает PDF и возвращает текст,
    разделённый по страницам.
    """

    doc = fitz.open(path)

    pages_text = []

    for page in doc:
        text = page.get_text("text")

        # минимальная нормализация
        text = text.replace("\r", "").strip()

        if text:
            pages_text.append(text)

    # разделяем страницы двойным переносом
    return "\n\n".join(pages_text)

In [ ]:
from docx import Document

def load_docx_as_text(path: str) -> str:
    """
    Загружает DOCX и возвращает текст
    с сохранением структуры параграфов.
    """

    doc = Document(path)

    paragraphs = []

    for p in doc.paragraphs:
        text = p.text.strip()

        if not text:
            continue

        paragraphs.append(text)

    return "\n\n".join(paragraphs)

In [ ]:
import os

def load_document(path: str) -> str:
    ext = os.path.splitext(path)[1].lower()

    if ext == ".pdf":
        return load_pdf_as_text(path)

    if ext == ".docx":
        return load_docx_as_text(path)

    raise ValueError(f"Unsupported format: {ext}")

# DOCUMENTS PARSING

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class Chunk:
    id: str
    doc_name: str
    section: str
    text: str
    level: str  # "chapter" | "article" | "point"
    parent_id: Optional[str] = None

In [183]:
import re

PATTERNS = {
    "chapter": re.compile(r"^(Глава\s+\d+\.?.*)$", re.MULTILINE),
    "section": re.compile(r"^(Раздел\s+[IVXLCDM\d]+\.?.*)$", re.MULTILINE),
    "article": re.compile(r"^(Статья\s+\d+(\.\d+)?\.?.*)$", re.MULTILINE),
    "point": re.compile(r"^(\d+(\.\d+)*\.)\s", re.MULTILINE),
}

In [ ]:
def parse_directory(directory: str):

    all_chunks = []

    paths = load_documents_from_dir(directory)

    for path in paths:

        print(f"Parsing {path}")

        try:

            text = load_document(str(path))

            chunks = parse_document(
                doc_name=path.name,
                text=text
            )

            all_chunks.extend(chunks)

        except Exception as e:

            print(
                f"Failed {path}: {e}"
            )

    return all_chunks

In [184]:
def extract_structured_headers(text: str):

    headers = []

    for level, pattern in PATTERNS.items():

        for m in pattern.finditer(text):

            headers.append({
                "level": level,
                "title": m.group(1).strip(),
                "start": m.start()
            })

    return sorted(headers, key=lambda x: x["start"])

In [ ]:
LEVEL_ORDER = ["section", "chapter", "article", "point"]

def _update_stack(stack, header, node_id, level):

    # убираем всё что глубже или равно текущему уровню
    while stack:
        last_level = stack[-1]["level"]

        if LEVEL_ORDER.index(last_level) >= LEVEL_ORDER.index(level):
            stack.pop()
        else:
            break

    stack.append({
        "id": node_id,
        "level": level
    })

In [186]:
import uuid

def build_chunk_graph(doc_name: str, text: str):

    headers = extract_structured_headers(text)

    if not headers:
        return [
            Chunk(
                id=str(uuid.uuid4()),
                doc_name=doc_name,
                section="DOCUMENT",
                text=text
            )
        ]

    chunks = []
    stack = []  # хранит текущую иерархию

    for i, h in enumerate(headers):

        start = h["start"]
        end = headers[i + 1]["start"] if i + 1 < len(headers) else len(text)

        content = text[start:end].strip()

        node_id = str(uuid.uuid4())

        # определяем parent
        parent_id = stack[-1]["id"] if stack else None

        chunk = Chunk(
            id=node_id,
            doc_name=doc_name,
            level=h["level"],
            section=h["title"],
            text=content,
            parent_id=parent_id
        )

        chunks.append(chunk)

        # обновляем стек по уровню
        _update_stack(stack, h, node_id, h["level"])

    return chunks

In [187]:
def expand_path(chunk, chunks_by_id):

    path = []
    seen_levels = set()

    current = chunk

    while current:

        # берём только 1 узел каждого уровня
        if current.level not in seen_levels:

            path.append(current)
            seen_levels.add(current.level)

        if not current.parent_id:
            break

        current = chunks_by_id.get(current.parent_id)

    return list(reversed(path))

In [188]:
def build_chunks_by_id(chunks):

    return {
        chunk.id: chunk
        for chunk in chunks
    }

In [ ]:
from collections import defaultdict

def build_children_by_id(chunks):

    children = defaultdict(list)

    for chunk in chunks:

        if chunk.parent_id:

            children[
                chunk.parent_id
            ].append(chunk.id)

    return dict(children)

In [190]:
test_path = "/Users/z123/Desktop/rag-agent/personal_data.pdf"

In [191]:
text = load_document(test_path)

chunks = build_chunk_graph(doc_name=test_path.split('/')[-1], text = text)

# SPARSE RETRIEVAL

In [193]:
import re
import pymorphy3

morph = pymorphy3.MorphAnalyzer()

def normalize(text):

    words = re.findall(
        r"\w+",
        text.lower()
    )

    return [
        morph.parse(word)[0].normal_form
        for word in words
    ]

In [194]:
! pip install pymorphy3 rank-bm25


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [195]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [
    normalize(chunk.text)
    for chunk in chunks
]

bm25 = BM25Okapi(tokenized_chunks)

In [196]:
def sparse_search(query, bm25, chunks, k):

    tokens = normalize(query)

    scores = bm25.get_scores(tokens)

    ranked = sorted(
        enumerate(scores),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for idx, score in ranked[:k]:

        results.append({
            "score": float(score),
            "chunk": chunks[idx]
        })

    return results

# DENSE RETRIEVAL

In [ ]:
! pip install sentence-transformers

In [200]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("intfloat/multilingual-e5-small")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [201]:
def embed_query(text: str):
    return model.encode(
        f"query: {text}",
        normalize_embeddings=True
    )


def embed_doc(text: str):
    return model.encode(
        f"passage: {text}",
        normalize_embeddings=True
    )

In [202]:
def build_dense_index(chunks):

    texts = [
        f"passage: {c.section}\n{c.text}"
        for c in chunks
    ]

    embeddings = model.encode(
        texts,
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=True
    )

    return np.array(embeddings)

In [203]:
def dense_search(query, chunks, dense_index, k=5):

    chunk_embeddings = dense_index

    q = embed_query(query)

    scores = np.dot(chunk_embeddings, q)

    top_idx = np.argsort(scores)[::-1][:k]

    return [
        {
            "score": float(scores[i]),
            "chunk": chunks[i]
        }
        for i in top_idx
    ]

In [204]:
chunks_by_id = build_chunks_by_id(chunks)
children_by_id = build_children_by_id(chunks)
dense_index = build_dense_index(chunks)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

In [206]:
# dense_ids = [result['chunk'].id for result in dense_search(test_query)]

In [ ]:
chunks = parse_directory(
    "./documents"
)

chunks_by_id = build_chunks_by_id(
    chunks
)

children_by_id = build_children_by_id(
    chunks
)

dense_index = build_dense_index(
    chunks
)

tokenized_chunks = [
    normalize(
        c.section + "\n" + c.text
    )
    for c in chunks
]

bm25 = BM25Okapi(
    tokenized_chunks
)

# TOOLS

In [215]:
def rrf(rank, k=60):
    return 1.0 / (k + rank)

def hybrid_search(
    query,
    bm25,
    chunks,
    dense_index,
    top_k
):

    sparse = sparse_search(
        query = query,
        bm25=bm25,
        chunks=chunks,
        k=top_k
    )

    dense = dense_search(
        query=query,
        chunks=chunks,
        dense_index=dense_index,
        k=top_k
    )

    scores = {}

    for rank, item in enumerate(
        sparse,
        start=1
    ):

        cid = item["chunk"].id

        scores[cid] = (
            scores.get(cid, 0)
            + rrf(rank)
        )

    for rank, item in enumerate(
        dense,
        start=1
    ):

        cid = item["chunk"].id

        scores[cid] = (
            scores.get(cid, 0)
            + rrf(rank)
        )

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for cid, score in ranked[:top_k]:

        results.append({
            "chunk": chunks_by_id[cid],
            "score": score
        })

    return results

In [216]:
def retrieve(query: str, top_k: int = 5):

    results = hybrid_search(
        query=query,
        bm25=bm25,
        chunks=chunks,
        dense_index=dense_index,
        top_k=top_k
    )

    return [
        {
            "chunk_id": r["chunk"].id,
            "section": r["chunk"].section,
            "score": round(r["score"], 4)
        }
        for r in results
    ]

In [217]:
def read_chunk(chunk_id: str):

    chunk = chunks_by_id.get(chunk_id)

    if chunk is None:
        return {
            "error": "chunk not found"
        }

    return {
        "chunk_id": chunk.id,
        "section": chunk.section,
        "text": chunk.text
    }

In [218]:
def expand_path(chunk_id: str):

    chunk = chunks_by_id.get(chunk_id)

    if chunk is None:
        return []

    chapter = None
    article = None
    point = None

    current = chunk

    while current:

        if current.level == "chapter":
            chapter = current.section

        elif current.level == "article":
            article = current.section

        elif current.level == "point":
            point = current.section

        if not current.parent_id:
            break

        current = chunks_by_id.get(
            current.parent_id
        )

    result = []

    if chapter:
        result.append(chapter)

    if article:
        result.append(article)

    if point:
        result.append(point)

    return result

In [219]:
def get_neighbors(
    chunk_id: str,
    window: int = 1
):

    chunk = chunks_by_id.get(chunk_id)

    if chunk is None:
        return []

    parent_id = chunk.parent_id

    if parent_id is None:
        return []

    siblings = children_by_id.get(
        parent_id,
        []
    )

    try:
        idx = siblings.index(chunk_id)

    except ValueError:
        return []

    start = max(
        0,
        idx - window
    )

    end = min(
        len(siblings),
        idx + window + 1
    )

    result = []

    for cid in siblings[start:end]:

        c = chunks_by_id[cid]

        result.append({
            "chunk_id": c.id,
            "section": c.section,
            "text": c.text
        })

    return result

In [220]:
TOOLS = {
    "retrieve": retrieve,
    "read_chunk": read_chunk,
    "expand_path": expand_path,
    "get_neighbors": get_neighbors,
}

In [221]:
import json

def execute_tool(tool_call):

    name = tool_call.function.name

    arguments = json.loads(
        tool_call.function.arguments
    )

    tool = TOOLS[name]

    return tool(**arguments)

In [222]:
retrieve(test_query)

[{'chunk_id': '47096f9a-b19d-4856-9ddd-d10d77c612ab',
  'section': '3.',
  'score': 0.0325},
 {'chunk_id': 'ecf5bcad-c8f3-49b6-8b11-afe847cd36d5',
  'section': '11.',
  'score': 0.032},
 {'chunk_id': 'e12c30b5-d9ad-43bf-bae1-5f528c7f0705',
  'section': '2.',
  'score': 0.0318},
 {'chunk_id': '33013faf-64c6-4301-b4c6-5d78579ab9a4',
  'section': '4.',
  'score': 0.0317},
 {'chunk_id': '3387b7ec-3f3d-4df4-99f5-8a48e38ac1b0',
  'section': 'Глава 1. ОБЩИЕ ПОЛОЖЕНИЯ',
  'score': 0.0154}]

# ReAct CYCLE

In [267]:
SYSTEM_PROMPT = """
Ты агент по нормативным документам.

Для ответа используй инструменты.

Алгоритм:
1. Сначала вызови retrieve().
2. Затем expand_path().
3. Затем read_chunk().
4. При необходимости вызови get_neighbors().
5. После получения достаточной информации дай ответ.

ВАЖНО:
Всегда отвечай ТОЛЬКО на русском языке.
Никогда не используй китайский, английский или другие языки.
Все ответы, рассуждения и объяснения должны быть исключительно на русском языке.
Если цитируешь документ, сохраняй язык оригинала документа.

Никогда не придумывай факты.
Всегда опирайся на результаты инструментов.

"""

In [268]:
tool_schemas = [
    {
        "type": "function",
        "function": {
            "name": "retrieve",
            "description": (
                "Search relevant chunks."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string"
                    }
                },
                "required": ["query"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "read_chunk",
            "description": (
                "Read full chunk text."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "chunk_id": {
                        "type": "string"
                    }
                },
                "required": ["chunk_id"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "expand_path",
            "description": (
                "Get chapter/article/point path."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "chunk_id": {
                        "type": "string"
                    }
                },
                "required": ["chunk_id"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "get_neighbors",
            "description": (
                "Get neighboring chunks."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "chunk_id": {
                        "type": "string"
                    },
                    "window": {
                        "type": "integer"
                    }
                },
                "required": ["chunk_id"]
            }
        }
    }
]

In [269]:
def run_agent(
    client,
    question: str,
    max_steps: int = 10
):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": question
        }
    ]

    for step in range(max_steps):

        response = client.chat.completions.create(
            model="qwen2.5:7b",
            messages=messages,
            tools=tool_schemas,
            temperature = 0.1
        )

        msg = response.choices[0].message

        if not msg.tool_calls:

            return msg.content

        messages.append(msg)

        for tc in msg.tool_calls:

            result = execute_tool(tc)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(
                        result,
                        ensure_ascii=False
                    )
                }
            )

        if step == max_steps - 1:
            messages.append({
                "role": "system",
                "content": (
                    "You have enough information. "
                    "Provide the final answer now."
                )
            })

    return msg.content

In [270]:
! pip install openai


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [271]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [ ]:
result = run_agent(client, "расскажи мне как защищают мои персональные данные")

In [274]:
result

'Первый шаг в защите персональных данных — это обеспечение соблюдения требований законодательства Российской Федерации. Уполномоченным органом по защите прав субъектов персональных данных является федеральный орган исполнительной власти, который осуществляет контроль и надзор за обработкой персональных данных в соответствии с требованиями законодательства о персональных данных.\n\nДля более подробного понимания процесса защиты персональных данных рекомендуется изучить дополнительные разделы документа.'